# Build the US Macro Dashboard - self-contained

This folder is fully portable. It contains:
- **`build_dashboard.py`** - all the build source in one file (regenerates the indicator catalog from the data, then builds the dashboard)
- **`all_macro_data.parquet`** - the only data file needed
- this notebook

**Run All Cells** (menu &#9656; Run &#9656; Run All Cells) to (re)build **`macro_dashboard.html`** right here and open it. No bash, no other files required.

In [ ]:
# ============================================================
#  CONFIG
# ============================================================
OPEN_WHEN_DONE    = True    # open the finished dashboard in your browser
AUTO_INSTALL_DEPS = True    # pip-install pandas/pyarrow into this kernel if missing

import sys, subprocess, time
from pathlib import Path

# the notebook, build_dashboard.py and all_macro_data.parquet all live together
def _find_dir():
    start = Path.cwd().resolve()
    for d in [start, *start.parents]:
        if (d / "build_dashboard.py").exists() and (d / "all_macro_data.parquet").exists():
            return d
    raise FileNotFoundError(
        "Keep this notebook in the same folder as build_dashboard.py and "
        "all_macro_data.parquet, then re-run."
    )

HERE     = _find_dir()
BUILDER  = HERE / "build_dashboard.py"
PARQUET  = HERE / "all_macro_data.parquet"
HTML_OUT = HERE / "macro_dashboard.html"
print("Folder  :", HERE)
print("Builder :", "ok" if BUILDER.exists() else "MISSING")
print("Data    :", "ok" if PARQUET.exists() else "MISSING")

def _missing(mod):
    try:
        __import__(mod); return False
    except Exception:
        return True
need = [m for m in ("pandas", "pyarrow") if _missing(m)]
if need and AUTO_INSTALL_DEPS:
    print("Installing missing deps:", need)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *need], check=True)
elif need:
    print("Missing deps:", need, "-> run:", sys.executable, "-m pip install", *need)
else:
    print("Deps    : pandas + pyarrow ok")

In [ ]:
# ============================================================
#  BUILD  ->  regenerates the catalog from the parquet, then the dashboard
# ============================================================
from datetime import datetime
t0 = time.time()
print("Building (catalog + dashboard) ...")
p = subprocess.run([sys.executable, str(BUILDER)], cwd=str(HERE),
                   capture_output=True, text=True)
if p.stdout.strip():
    print(p.stdout.rstrip())
if p.returncode != 0:
    print("FAILED:")
    print(p.stderr.rstrip() or "(no stderr)")
    raise RuntimeError("build_dashboard.py exited with code " + str(p.returncode))
print("Done in", round(time.time() - t0, 1), "s")

if not HTML_OUT.exists():
    raise FileNotFoundError("Expected output not found: " + str(HTML_OUT))
size_mb = HTML_OUT.stat().st_size / 1e6
url = HTML_OUT.resolve().as_uri()
print()
print("Dashboard:", HTML_OUT, "(" + str(round(size_mb, 2)) + " MB)")
print(url)
try:
    from IPython.display import display, HTML
    display(HTML('<a href="' + url + '" target="_blank" '
                 'style="font:600 13px system-ui;color:#2563eb">&#8599; Open macro_dashboard.html</a>'))
except Exception:
    pass
if OPEN_WHEN_DONE:
    import webbrowser
    webbrowser.open(url)
    print("Opened in your default browser.")